In [ ]:
"""
Soil + Crop Recommendation Pipeline
------------------------------------
Predicts (or accepts) soil type, auto-fills or accepts NPK, then predicts the
best-suited crop using a trained model.

Assumes these objects already exist in scope (loaded elsewhere in your notebook):
    model_custom   - trained soil image classifier (torch.nn.Module)
    model          - trained crop recommendation model (torch.nn.Module)
    scaler         - fitted sklearn scaler for [N, P, K, temp, humidity, ph, rainfall]
    le             - fitted LabelEncoder for crop classes
"""

import os
import numpy as np
import torch
from torchvision import transforms
from PIL import Image

# ============================================================
# CONFIG
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SOIL_TYPE_MAP = {
    'alluvial': 0,
    'black':    1,
    'red':      2,
    'laterite': 3,
}
SOIL_CLASS_NAMES = list(SOIL_TYPE_MAP.keys())

# NOTE: the crop model's one-hot soil vector must match the width it was
# TRAINED with. If it was trained on only 4 soil types, set this to 4.
# Keeping it at 8 (as in the original code) only makes sense if the model
# reserves 4 extra unused soil-class slots. Confirm this against your
# training script before relying on predictions.
TOTAL_SOIL_CLASSES = 8

NPK_DEFAULTS = {
    'alluvial': (90, 35, 45),
    'black':    (100, 40, 50),
    'red':      (50, 25, 70),
    'laterite': (40, 22, 60),
}

SOIL_IMAGE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])

# Rough physical bounds used only to catch typos, not to enforce agronomy
VALID_RANGES = {
    "Nitrogen (N)":      (0, 300),
    "Phosphorus (P)":    (0, 300),
    "Potassium (K)":     (0, 300),
    "Temperature (°C)":  (-10, 60),
    "Humidity (%)":      (0, 100),
    "Soil pH":           (0, 14),
    "Rainfall (mm)":     (0, 5000),
}


# ============================================================
# INPUT HELPERS
# ============================================================
def ask_choice(prompt, choices):
    """Repeatedly prompt until the answer is one of `choices` (case-insensitive)."""
    choices_lower = [c.lower() for c in choices]
    while True:
        val = input(prompt).strip().lower()
        if val in choices_lower:
            return val
        print(f"⚠️  Please enter one of: {', '.join(choices)}")


def ask_float(prompt, label):
    """Repeatedly prompt for a float within VALID_RANGES[label]."""
    lo, hi = VALID_RANGES[label]
    while True:
        raw = input(prompt).strip()
        try:
            val = float(raw)
        except ValueError:
            print("⚠️  Please enter a numeric value.")
            continue
        if not (lo <= val <= hi):
            print(f"⚠️  {label} should be between {lo} and {hi}. Try again.")
            continue
        return val


# ============================================================
# SOIL PREDICTION
# ============================================================
def predict_soil(image_path, image_model, transform, class_names):
    """Run the soil image classifier and return the predicted class name."""
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        outputs = image_model(tensor)
        pred_idx = torch.argmax(outputs, dim=1).item()
    return class_names[pred_idx]


def get_soil_type(image_model):
    """Ask the user whether they know the soil type, or classify it from an image."""
    knows_soil = ask_choice("🧭 Do you know the soil type? (yes/no): ", ["yes", "no"])

    if knows_soil == "yes":
        return ask_choice(
            "🟤 Enter soil type (alluvial / black / red / laterite): ",
            SOIL_CLASS_NAMES,
        )

    # knows_soil == "no" -> classify from image
    while True:
        image_path = input("📁 Enter the full path to the soil image: ").strip()
        if not os.path.isfile(image_path):
            print("❌ Invalid image path. Try again.")
            continue
        try:
            soil = predict_soil(image_path, image_model, SOIL_IMAGE_TRANSFORM, SOIL_CLASS_NAMES)
        except Exception as e:
            print(f"❌ Could not process image ({e}). Try again.")
            continue
        print(f"\n🧠 Predicted Soil Type from Image: {soil.capitalize()}")
        return soil


# ============================================================
# NPK
# ============================================================
def get_npk_values(soil):
    auto_fill = ask_choice(
        "\n🔄 Auto-fill NPK based on soil type? (yes/no): ", ["yes", "no"]
    )
    if auto_fill == "yes":
        n, p, k = NPK_DEFAULTS[soil]
        print(f"\n📦 Auto-filled NPK values for '{soil.capitalize()}':")
        print(f"   🧪 Nitrogen (N): {n}")
        print(f"   🧪 Phosphorus (P): {p}")
        print(f"   🧪 Potassium (K): {k}")
        return float(n), float(p), float(k)

    n = ask_float("Enter Nitrogen (N): ", "Nitrogen (N)")
    p = ask_float("Enter Phosphorus (P): ", "Phosphorus (P)")
    k = ask_float("Enter Potassium (K): ", "Potassium (K)")
    return n, p, k


def get_environment_inputs():
    temperature = ask_float("\n🌡️ Enter Temperature (°C): ", "Temperature (°C)")
    humidity = ask_float("💧 Enter Humidity (%): ", "Humidity (%)")
    ph = ask_float("🧪 Enter soil pH: ", "Soil pH")
    rainfall = ask_float("🌧️ Enter Rainfall (mm): ", "Rainfall (mm)")
    return temperature, humidity, ph, rainfall


# ============================================================
# CROP PREDICTION
# ============================================================
def build_crop_input(n, p, k, temperature, humidity, ph, rainfall, soil, scaler):
    raw = np.array([[n, p, k, temperature, humidity, ph, rainfall]])
    scaled = scaler.transform(raw)

    one_hot = np.zeros(TOTAL_SOIL_CLASSES, dtype=np.float32)
    one_hot[SOIL_TYPE_MAP[soil]] = 1.0

    combined = np.concatenate([scaled, one_hot.reshape(1, -1)], axis=1).astype(np.float32)
    return torch.tensor(combined, dtype=torch.float32, device=DEVICE)


def predict_crop(crop_model, input_tensor, label_encoder, top_k=3):
    with torch.no_grad():
        output = crop_model(input_tensor)
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]
    top_indices = probs.argsort()[::-1][:top_k]
    return [(label_encoder.classes_[i], probs[i]) for i in top_indices]


# ============================================================
# MAIN
# ============================================================
def run_recommendation(model_custom, model, scaler, le):
    model_custom.eval().to(DEVICE)
    model.eval().to(DEVICE)

    soil = get_soil_type(model_custom)
    n, p, k = get_npk_values(soil)
    temperature, humidity, ph, rainfall = get_environment_inputs()

    input_tensor = build_crop_input(n, p, k, temperature, humidity, ph, rainfall, soil, scaler)
    results = predict_crop(model, input_tensor, le, top_k=3)

    print("\n📈 Prediction Results:")
    for crop, prob in results:
        print(f"  🌿 {crop}: {prob:.4f}")
    print(f"\n✅ Best Suited Crop: {results[0][0]}")

    return results


if __name__ == "__main__":
    # Expects model_custom, model, scaler, le to already be defined in this session
    run_recommendation(model_custom, model, scaler, le)

SyntaxError: invalid syntax (2367018055.py, line 1)